# Preparação e Integração dos Dados da EV.G

Este notebook documenta a preparação dos dados utilizados no projeto de **Sistema Híbrido de Recomendação de Cursos para Desenvolvimento Profissional**, desenvolvido a partir do catálogo atual e do histórico de matrículas da Escola Virtual de Governo (EV.G).

O objetivo desta etapa é compreender a estrutura das bases, integrar catálogo e histórico, tratar diferenças de identificação dos cursos, selecionar as variáveis relevantes e gerar uma base processada adequada para as etapas de análise exploratória e modelagem.


## 1. Configuração do ambiente

A preparação foi realizada localmente em Python, utilizando principalmente **Pandas**, **NumPy** e **DuckDB**. O DuckDB foi utilizado para consultar diretamente o arquivo histórico de matrículas, que possui grande volume de dados, evitando seu carregamento integral em memória.

> **Observação sobre os caminhos dos arquivos:** este notebook foi desenvolvido no ambiente local da autora. Os caminhos abaixo correspondem à estrutura utilizada durante o desenvolvimento. Para executar o notebook em outro computador, é necessário ajustar os caminhos dos arquivos de entrada e da pasta de saída de acordo com a estrutura local.


In [ ]:
import sys

print("Python:", sys.version)

In [ ]:
import pandas as pd
import numpy as np

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

In [ ]:
%pip install duckdb

In [ ]:
import duckdb

print("DuckDB:", duckdb.__version__)

## 2. Definição dos arquivos e conexão com o DuckDB

O arquivo histórico de matrículas e o catálogo de cursos são mantidos localmente. O histórico contém milhões de registros, enquanto o catálogo representa os cursos disponibilizados pela EV.G no momento da coleta.


In [ ]:
from pathlib import Path

# Pasta do projeto
pasta_projeto = Path("..")

# Arquivos
arquivo_matriculas = pasta_projeto / "data set" / "20260902_escolavirtual_dadosabertos_matriculas_utf8.csv"
arquivo_catalogo = pasta_projeto / "data set" / "catalogo.csv"

print("Matrículas encontrado:", arquivo_matriculas.exists())
print("Catálogo encontrado:", arquivo_catalogo.exists())

In [ ]:
con = duckdb.connect()

estrutura_matriculas = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_csv_auto('{arquivo_matriculas.as_posix()}')
""").df()

estrutura_matriculas

## 3. Conhecimento inicial do histórico de matrículas

Antes da integração, são verificadas a dimensão da base, a quantidade de usuários e cursos, o período coberto e a distribuição das situações de matrícula. Essa etapa permite compreender a estrutura geral do histórico antes de qualquer transformação.


In [ ]:
resumo_base = con.execute(f"""
    SELECT
        COUNT(*) AS total_matriculas,
        COUNT(DISTINCT codigo_pessoa) AS total_usuarios,
        COUNT(DISTINCT cod_curso) AS total_cursos,
        MIN(dt_matricula) AS primeira_matricula,
        MAX(dt_matricula) AS ultima_matricula,
        COUNT(*) FILTER (WHERE codigo_pessoa IS NULL) AS usuarios_nulos,
        COUNT(*) FILTER (WHERE cod_curso IS NULL) AS cursos_nulos,
        COUNT(*) FILTER (WHERE sit_matricula IS NULL) AS situacoes_nulas
    FROM read_csv_auto('{arquivo_matriculas.as_posix()}')
""").df()

resumo_base

In [ ]:
situacoes = con.execute(f"""
    SELECT
        sit_matricula,
        COUNT(*) AS quantidade,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentual
    FROM read_csv_auto('{arquivo_matriculas.as_posix()}')
    GROUP BY sit_matricula
    ORDER BY quantidade DESC
""").df()

situacoes

## 4. Carregamento e inspeção do catálogo atual

O catálogo é carregado separadamente para permitir a comparação entre os cursos atualmente disponíveis e os registros históricos.

O caminho abaixo foi mantido exatamente como utilizado no computador em que a preparação foi executada. Para reproduzir o notebook em outro ambiente, esse caminho deve ser alterado.


In [ ]:
caminho_catalogo = r"C:\Users\PAULO\Documents\Banco de dados\4 semestre\PROJETO APLICADO III\data set\catalogo.csv"

catalogo = con.execute(f"""
    SELECT *
    FROM read_csv_auto('{caminho_catalogo}')
""").df()

print("Dimensões do catálogo:", catalogo.shape)

print("\nColunas:")
print(catalogo.columns.tolist())

catalogo.head()

## 5. Integração entre catálogo e histórico

### 5.1 Verificação dos identificadores

Inicialmente, foi testada a possibilidade de relacionar diretamente `cod_curso`, presente no histórico, com `id_curso`, presente no catálogo. Os formatos dos identificadores são diferentes, portanto eles não constituem uma chave de integração direta.

Em seguida, a integração passou a considerar o nome do curso como elemento de correspondência.


In [ ]:
comparacao_cursos = con.execute(f"""
    SELECT
        COUNT(DISTINCT h.cod_curso) AS cursos_historico,
        COUNT(DISTINCT c.id_curso) AS cursos_catalogo,
        COUNT(DISTINCT CASE 
            WHEN c.id_curso IS NOT NULL THEN h.cod_curso 
        END) AS cursos_em_comum
    FROM read_csv_auto('{arquivo_matriculas.as_posix()}') h
    LEFT JOIN read_csv_auto('{caminho_catalogo}') c
        ON TRY_CAST(h.cod_curso AS BIGINT) = c.id_curso
""").df()

comparacao_cursos

In [ ]:
comparacao_nomes = con.execute(f"""
    WITH historico AS (
        SELECT DISTINCT
            cod_curso,
            nome_curso
        FROM read_csv_auto('{arquivo_matriculas.as_posix()}')
    ),
    catalogo_atual AS (
        SELECT DISTINCT
            id_curso,
            nome_curso
        FROM read_csv_auto('{caminho_catalogo}')
    )
    
    SELECT
        COUNT(DISTINCT h.cod_curso) AS cursos_historico,
        COUNT(DISTINCT c.id_curso) AS cursos_catalogo,
        COUNT(DISTINCT c.id_curso) FILTER (
            WHERE h.cod_curso IS NOT NULL
        ) AS cursos_catalogo_com_historico
    FROM catalogo_atual c
    LEFT JOIN historico h
        ON TRIM(LOWER(c.nome_curso)) = TRIM(LOWER(h.nome_curso))
""").df()

comparacao_nomes

In [ ]:
cobertura_catalogo = con.execute(f"""
    WITH historico AS (
        SELECT DISTINCT
            cod_curso,
            TRIM(LOWER(nome_curso)) AS nome_normalizado
        FROM read_csv_auto('{arquivo_matriculas.as_posix()}')
    ),
    catalogo_atual AS (
        SELECT DISTINCT
            id_curso,
            TRIM(LOWER(nome_curso)) AS nome_normalizado
        FROM read_csv_auto('{caminho_catalogo}')
    ),
    correspondencias AS (
        SELECT DISTINCT
            c.id_curso
        FROM catalogo_atual c
        INNER JOIN historico h
            ON c.nome_normalizado = h.nome_normalizado
    )
    
    SELECT
        (SELECT COUNT(DISTINCT cod_curso) FROM historico) AS cursos_historico,
        (SELECT COUNT(DISTINCT id_curso) FROM catalogo_atual) AS cursos_catalogo,
        (SELECT COUNT(*) FROM correspondencias) AS cursos_com_historico,
        ROUND(
            (SELECT COUNT(*) FROM correspondencias) * 100.0 /
            (SELECT COUNT(DISTINCT id_curso) FROM catalogo_atual),
            2
        ) AS cobertura_percentual
""").df()

cobertura_catalogo

### 5.2 Investigação das diferenças entre os nomes

A comparação inicial, baseada apenas em letras minúsculas e remoção de espaços nas extremidades, não encontrou correspondência para todos os cursos do catálogo. A inspeção dos títulos indicou diferenças pequenas de grafia, acentuação e pontuação.

Por isso, foi adotada uma normalização mais robusta dos nomes, com conversão para minúsculas, remoção de acentos, substituição de pontuação por espaços e eliminação de espaços duplicados.


In [ ]:
cursos_historicos = con.execute(f"""
    SELECT DISTINCT
        cod_curso,
        nome_curso
    FROM read_csv_auto('{arquivo_matriculas.as_posix()}')
    WHERE nome_curso IS NOT NULL
""").df()

print("Códigos históricos distintos:", cursos_historicos["cod_curso"].nunique())


In [ ]:
import re
import unicodedata

def normalizar_nome(texto):
    if pd.isna(texto):
        return None

    # minúsculas
    texto = texto.lower()

    # remove acentos
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(
        caractere for caractere in texto
        if not unicodedata.combining(caractere)
    )

    # substitui pontuação por espaço
    texto = re.sub(r"[^\w\s]", " ", texto)

    # remove espaços duplicados
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto

In [ ]:
catalogo_normalizado = catalogo.copy()
historico_normalizado = cursos_historicos.copy()

catalogo_normalizado["nome_normalizado"] = (
    catalogo_normalizado["nome_curso"].apply(normalizar_nome)
)

historico_normalizado["nome_normalizado"] = (
    historico_normalizado["nome_curso"].apply(normalizar_nome)
)

correspondencia_normalizada = catalogo_normalizado.merge(
    historico_normalizado,
    on="nome_normalizado",
    how="left",
    suffixes=("_catalogo", "_historico")
)

resumo_normalizacao = pd.DataFrame({
    "cursos_catalogo": [catalogo["id_curso"].nunique()],
    "cursos_com_correspondencia": [
        correspondencia_normalizada.loc[
            correspondencia_normalizada["cod_curso"].notna(),
            "id_curso"
        ].nunique()
    ]
})

resumo_normalizacao["sem_correspondencia"] = (
    resumo_normalizacao["cursos_catalogo"]
    - resumo_normalizacao["cursos_com_correspondencia"]
)

resumo_normalizacao

### 5.3 Verificação de colisões após a normalização

A normalização aumentou a cobertura da correspondência, mas também revelou grupos em que diferentes identificadores passaram a representar o mesmo título normalizado. Esses casos foram analisados antes da consolidação para evitar que registros semanticamente equivalentes fossem tratados como cursos diferentes.


In [ ]:
colisoes_catalogo = (
    catalogo_normalizado
    .groupby("nome_normalizado")
    .agg(
        quantidade_ids=("id_curso", "nunique"),
        quantidade_nomes=("nome_curso", "nunique")
    )
    .query("quantidade_ids > 1")
    .reset_index()
)

colisoes_historico = (
    historico_normalizado
    .groupby("nome_normalizado")
    .agg(
        quantidade_codigos=("cod_curso", "nunique"),
        quantidade_nomes=("nome_curso", "nunique")
    )
    .query("quantidade_nomes > 1")
    .reset_index()
)

print("Colisões no catálogo:", len(colisoes_catalogo))
print("Colisões de nomes no histórico:", len(colisoes_historico))

In [ ]:
colisoes_catalogo_detalhes = (
    catalogo_normalizado[
        catalogo_normalizado["nome_normalizado"].isin(
            colisoes_catalogo["nome_normalizado"]
        )
    ][
        ["id_curso", "nome_curso", "nome_normalizado"]
    ]
    .sort_values(["nome_normalizado", "id_curso"])
)

pd.set_option("display.max_colwidth", None)

colisoes_catalogo_detalhes

In [ ]:
detalhes_colisoes = catalogo_normalizado[
    catalogo_normalizado["nome_normalizado"].isin(
        colisoes_catalogo["nome_normalizado"]
    )
][[
    "id_curso",
    "nome_curso",
    "data_lancamento",
    "carga_horaria",
    "conteudista",
    "eixos_tematicos",
    "tipo_oferta"
]].sort_values(
    ["nome_curso", "data_lancamento"]
)

pd.set_option("display.max_colwidth", None)

detalhes_colisoes

### 5.4 Criação do identificador canônico

Foram identificados **1.041 registros de cursos no catálogo**. Após a padronização dos títulos, foram observados **12 grupos de registros semanticamente equivalentes associados a identificadores distintos**.

Para evitar que essas variações fossem interpretadas pelo sistema de recomendação como itens diferentes, foi criado o campo interno `id_curso_canonico`. Os identificadores originais são preservados para rastreabilidade. A consolidação resultou em **1.029 cursos conceituais**.


In [ ]:
total_catalogo = catalogo["id_curso"].nunique()

total_canonicos = (
    catalogo_normalizado["nome_normalizado"]
    .nunique()
)

print("Cursos originais no catálogo:", total_catalogo)
print("Cursos após canonicalização:", total_canonicos)
print("Registros consolidados:", total_catalogo - total_canonicos)

In [ ]:
catalogo_normalizado["id_curso_canonico"] = (
    pd.factorize(catalogo_normalizado["nome_normalizado"])[0] + 1
)

print(
    "Cursos canônicos:",
    catalogo_normalizado["id_curso_canonico"].nunique()
)

catalogo_normalizado[
    [
        "id_curso",
        "id_curso_canonico",
        "nome_curso",
        "nome_normalizado"
    ]
].head()

### 5.5 Mapeamento dos códigos históricos para os cursos canônicos

O histórico pode apresentar diferentes códigos e pequenas variações de nome para um mesmo curso. Por isso, os códigos históricos foram relacionados ao identificador canônico por meio do nome normalizado.

Também foi verificado se algum código histórico apontava para mais de um curso canônico. Essa validação é importante para garantir consistência antes da criação da base processada.


In [ ]:
mapa_cursos = (
    historico_normalizado[
        ["cod_curso", "nome_curso", "nome_normalizado"]
    ]
    .merge(
        catalogo_normalizado[
            ["id_curso_canonico", "nome_normalizado"]
        ].drop_duplicates(),
        on="nome_normalizado",
        how="left"
    )
)

print("Códigos históricos:", mapa_cursos["cod_curso"].nunique())

print(
    "Códigos históricos mapeados:",
    mapa_cursos.loc[
        mapa_cursos["id_curso_canonico"].notna(),
        "cod_curso"
    ].nunique()
)

print(
    "Códigos históricos sem correspondência:",
    mapa_cursos.loc[
        mapa_cursos["id_curso_canonico"].isna(),
        "cod_curso"
    ].nunique()
)

mapa_cursos.head()

In [ ]:
status_mapeamento = (
    mapa_cursos
    .groupby("cod_curso")
    .agg(
        qtd_nomes=("nome_normalizado", "nunique"),
        qtd_ids_canonicos=("id_curso_canonico", "nunique"),
        possui_correspondencia=(
            "id_curso_canonico",
            lambda x: x.notna().any()
        )
    )
    .reset_index()
)

print("Códigos históricos:", len(status_mapeamento))

print(
    "Com pelo menos uma correspondência:",
    status_mapeamento["possui_correspondencia"].sum()
)

print(
    "Sem nenhuma correspondência:",
    (~status_mapeamento["possui_correspondencia"]).sum()
)

print(
    "Com mais de um nome histórico:",
    (status_mapeamento["qtd_nomes"] > 1).sum()
)

print(
    "Apontando para mais de um curso canônico:",
    (status_mapeamento["qtd_ids_canonicos"] > 1).sum()
)

In [ ]:
mapa_codigos_canonicos = (
    mapa_cursos
    .dropna(subset=["id_curso_canonico"])
    [["cod_curso", "id_curso_canonico"]]
    .drop_duplicates()
    .copy()
)

mapa_codigos_canonicos["id_curso_canonico"] = (
    mapa_codigos_canonicos["id_curso_canonico"].astype("int64")
)

print("Linhas do mapa:", len(mapa_codigos_canonicos))
print(
    "Códigos históricos únicos:",
    mapa_codigos_canonicos["cod_curso"].nunique()
)
print(
    "Cursos canônicos alcançados pelo histórico:",
    mapa_codigos_canonicos["id_curso_canonico"].nunique()
)

mapa_codigos_canonicos.head()

In [ ]:
resultado_interacoes = con.execute(f"""
    SELECT
        COUNT(*) AS total_interacoes,
        COUNT(DISTINCT m.codigo_pessoa) AS total_usuarios,
        COUNT(DISTINCT mapa.id_curso_canonico) AS total_cursos
    FROM read_csv_auto('{arquivo_matriculas.as_posix()}') AS m
    INNER JOIN mapa_codigos_canonicos AS mapa
        ON m.cod_curso = mapa.cod_curso
""").df()

resultado_interacoes

In [ ]:
status_interacoes = con.execute(f"""
    SELECT
        m.sit_matricula,
        COUNT(*) AS quantidade,
        ROUND(
            COUNT(*) * 100.0 /
            SUM(COUNT(*)) OVER (),
            2
        ) AS percentual
    FROM read_csv_auto('{arquivo_matriculas.as_posix()}') AS m
    INNER JOIN mapa_codigos_canonicos AS mapa
        ON m.cod_curso = mapa.cod_curso
    GROUP BY m.sit_matricula
    ORDER BY quantidade DESC
""").df()

status_interacoes

## 6. Seleção das variáveis da base processada

A base histórica original possui 28 colunas. Para a base processada foram mantidas as variáveis necessárias para identificar usuário, curso e interação, além de atributos considerados úteis para a análise exploratória.

**Variáveis mantidas:**
- `cod_matricula`: identificador original da matrícula, mantido para rastreabilidade;
- `dt_matricula`: data da interação;
- `cod_curso` e `nome_curso`: identificação histórica do curso;
- `carga_horaria`, `tematica` e `dt_fim`: características úteis para análise;
- `sit_matricula`: situação da matrícula;
- `codigo_pessoa`: identificador do usuário;
- `sexo` e `idade`: mantidos para análise exploratória, sem definição prévia de uso como atributos do modelo;
- `instituicao`, `poder`, `esfera` e `uf_servidor`: mantidos para análise contextual;
- `id_curso_canonico`: identificador interno do curso conceitual.

**Variáveis não incluídas na base processada:** `cod_turma`, `nome_turma`, `modalidade_turma`, `dt_inicio_insc`, `dt_fim_insc`, `dt_inicio`, `conteudista`, `deficiencia`, `raca`, `nacionalidade`, `uf_pessoa`, `municipio_pessoa` e `municipio_servidor`.

A exclusão foi feita considerando o objetivo do recomendador, redundância, granularidade da informação e necessidade de limitar o uso de atributos pessoais que não são essenciais ao projeto.


### 6.1 Validação de uma variável temporal redundante

Durante a seleção das colunas, `dt_inicio` foi comparada com `dt_matricula`. A verificação global mostrou que os dois campos possuem o mesmo valor em todos os registros do histórico analisado. Por esse motivo, `dt_inicio` não foi mantida na base processada.


In [ ]:
comparacao_datas = con.execute(f"""
    SELECT
        COUNT(*) AS total,
        COUNT(*) FILTER (
            WHERE dt_inicio = dt_matricula
        ) AS iguais,
        COUNT(*) FILTER (
            WHERE dt_inicio != dt_matricula
        ) AS diferentes,
        COUNT(*) FILTER (
            WHERE dt_inicio IS NULL
        ) AS inicio_nulo
    FROM read_csv_auto('{arquivo_matriculas.as_posix()}')
""").df()

comparacao_datas

## 7. Construção da base processada

A base processada contém apenas interações de cursos históricos que puderam ser associados aos cursos canônicos do catálogo atual. Todos os cinco status de matrícula são preservados nesta etapa.

Nenhuma consolidação de interações repetidas entre usuário e curso é realizada aqui. Essa decisão será tomada posteriormente, após a análise exploratória, de acordo com a estratégia adotada para o sistema de recomendação.


In [ ]:
from pathlib import Path

pasta_processados = pasta_projeto / "data" / "processed"
pasta_processados.mkdir(parents=True, exist_ok=True)

arquivo_interacoes = (
    pasta_processados / "interacoes_cursos_atuais.parquet"
)

arquivo_interacoes

In [ ]:
con.execute(f"""
    COPY (
        SELECT
            m.cod_matricula,
            m.dt_matricula,
            m.cod_curso,
            m.nome_curso,
            m.carga_horaria,
            m.dt_fim,
            m.tematica,
            m.sit_matricula,
            m.codigo_pessoa,
            m.sexo,
            m.idade,
            m.instituicao,
            m.poder,
            m.esfera,
            m.uf_servidor,
            mapa.id_curso_canonico
        FROM read_csv_auto('{arquivo_matriculas.as_posix()}') AS m
        INNER JOIN mapa_codigos_canonicos AS mapa
            ON m.cod_curso = mapa.cod_curso
    )
    TO '{arquivo_interacoes.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

## 8. Validação da base processada

Após a geração do arquivo Parquet, são verificadas suas dimensões, duplicidades, comportamento do identificador de matrícula e valores ausentes. O objetivo é confirmar que a transformação preservou as interações esperadas e identificar limitações relevantes antes da EDA.


In [ ]:
validacao_parquet = con.execute(f"""
    SELECT
        COUNT(*) AS total_interacoes,
        COUNT(DISTINCT codigo_pessoa) AS total_usuarios,
        COUNT(DISTINCT id_curso_canonico) AS total_cursos
    FROM read_parquet('{arquivo_interacoes.as_posix()}')
""").df()

validacao_parquet

### 8.1 Duplicatas exatas


In [ ]:
duplicatas_exatas = con.execute(f"""
    SELECT
        COUNT(*) AS total_linhas_duplicadas
    FROM (
        SELECT
            *,
            COUNT(*) AS quantidade
        FROM read_parquet('{arquivo_interacoes.as_posix()}')
        GROUP BY ALL
        HAVING COUNT(*) > 1
    )
""").df()

duplicatas_exatas

### 8.2 Unicidade de `cod_matricula`

Embora `cod_matricula` pareça inicialmente um candidato a identificador único da interação, a validação mostra que existem valores repetidos. Os registros envolvidos são investigados antes de qualquer exclusão.


In [ ]:
validacao_matriculas = con.execute(f"""
    SELECT
        COUNT(*) AS total_registros,
        COUNT(DISTINCT cod_matricula) AS matriculas_unicas,
        COUNT(*) - COUNT(DISTINCT cod_matricula) AS diferenca
    FROM read_parquet('{arquivo_interacoes.as_posix()}')
""").df()

validacao_matriculas

In [ ]:
analise_codigos_repetidos = con.execute(f"""
    SELECT
        cod_matricula,
        COUNT(*) AS qtd_registros,
        COUNT(DISTINCT codigo_pessoa) AS qtd_usuarios,
        COUNT(DISTINCT cod_curso) AS qtd_cursos,
        COUNT(DISTINCT dt_matricula) AS qtd_datas
    FROM read_parquet('{arquivo_interacoes.as_posix()}')
    GROUP BY cod_matricula
    HAVING COUNT(*) > 1
    ORDER BY qtd_registros DESC
""").df()

analise_codigos_repetidos

**Resultado da validação do identificador de matrícula:** foram identificados **207 valores repetidos** em `cod_matricula`, envolvendo **414 registros**. Em todos os casos, cada código repetido está associado a dois usuários, dois cursos e duas datas de matrícula distintos. Além disso, não foram encontradas linhas integralmente duplicadas.

Dessa forma, os registros foram mantidos e `cod_matricula` **não é considerado uma chave única global** da base. Para o sistema de recomendação, `codigo_pessoa` identifica o usuário e `id_curso_canonico` identifica o curso; a forma de consolidar múltiplas interações do mesmo usuário com o mesmo curso será definida na etapa de modelagem.


### 8.3 Valores ausentes

A quantidade de valores nulos é verificada em todas as 16 colunas da base processada. Os campos essenciais para a interação — usuário, curso, situação e data — devem ser avaliados separadamente dos atributos contextuais, pois a ausência em campos institucionais pode refletir a própria forma de coleta dos dados e não necessariamente um erro.


In [ ]:
nulos = con.execute(f"""
    SELECT
        COUNT(*) AS total_registros,

        COUNT(*) FILTER (WHERE cod_matricula IS NULL) AS cod_matricula,
        COUNT(*) FILTER (WHERE dt_matricula IS NULL) AS dt_matricula,
        COUNT(*) FILTER (WHERE cod_curso IS NULL) AS cod_curso,
        COUNT(*) FILTER (WHERE nome_curso IS NULL) AS nome_curso,
        COUNT(*) FILTER (WHERE carga_horaria IS NULL) AS carga_horaria,
        COUNT(*) FILTER (WHERE dt_fim IS NULL) AS dt_fim,
        COUNT(*) FILTER (WHERE tematica IS NULL) AS tematica,
        COUNT(*) FILTER (WHERE sit_matricula IS NULL) AS sit_matricula,
        COUNT(*) FILTER (WHERE codigo_pessoa IS NULL) AS codigo_pessoa,
        COUNT(*) FILTER (WHERE sexo IS NULL) AS sexo,
        COUNT(*) FILTER (WHERE idade IS NULL) AS idade,
        COUNT(*) FILTER (WHERE instituicao IS NULL) AS instituicao,
        COUNT(*) FILTER (WHERE poder IS NULL) AS poder,
        COUNT(*) FILTER (WHERE esfera IS NULL) AS esfera,
        COUNT(*) FILTER (WHERE uf_servidor IS NULL) AS uf_servidor,
        COUNT(*) FILTER (WHERE id_curso_canonico IS NULL) AS id_curso_canonico

    FROM read_parquet('{arquivo_interacoes.as_posix()}')
""").df()

nulos

### 8.4 Tamanho do arquivo processado


In [ ]:
tamanho_mb = arquivo_interacoes.stat().st_size / (1024 ** 2)
print(f"{tamanho_mb:.2f} MB")

## 9. Resultado da preparação

A preparação resultou em uma base processada em formato Parquet com **21.363.002 interações**, **6.444.712 usuários** e **1.029 cursos canônicos** associados ao catálogo atual.

As principais decisões desta etapa foram:

- utilização do DuckDB para processamento eficiente do histórico de grande volume;
- integração entre histórico e catálogo por títulos normalizados, devido à incompatibilidade entre os identificadores originais;
- criação de um identificador canônico para consolidar registros semanticamente equivalentes de cursos;
- preservação dos cinco status de matrícula, sem classificá-los antecipadamente como feedback positivo ou negativo;
- seleção de 16 variáveis para a base processada;
- manutenção das interações repetidas entre usuário e curso para investigação posterior;
- constatação de que `cod_matricula` não funciona como chave única global;
- geração do arquivo `interacoes_cursos_atuais.parquet`, que será utilizado na análise exploratória e na preparação do modelo de recomendação.

A etapa seguinte do projeto será realizada em um notebook separado, dedicado à **Análise Exploratória dos Dados (EDA)**.
